# Image Deblurring using Deep Learning — Restore Blurred Photographs
**AIML Final Year Project Notebook**

---
### Project Overview
Image blur caused by **camera shake (Motion Blur)** or **lens misfocus (Defocus Blur)** degrades visual quality and destroys high-frequency edge details. Mathematically, the degradation process is modeled as:
$$I_{\text{blur}} = K * I_{\text{sharp}} + \eta$$
where $K$ is an unknown Point Spread Function (PSF) kernel, $*$ denotes 2D spatial convolution, and $\eta$ is additive sensor noise.

In this notebook, we design, train, and evaluate a custom **Residual Channel-Attention U-Net (`DeblurUNet`)** that learns the inverse mapping by predicting the high-frequency residual detail:
$$\hat{I}_{\text{sharp}} = \text{clamp}\big(I_{\text{blur}} + f_{\theta}(I_{\text{blur}}),\; 0,\; 1\big)$$

In [ ]:
import os
import sys
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

# Ensure local modules are importable
PROJECT_DIR = Path.cwd()
if (PROJECT_DIR / 'image_deblurring_project').exists():
    PROJECT_DIR = PROJECT_DIR / 'image_deblurring_project'
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from dataset import (
    create_motion_blur_kernel,
    create_defocus_disk_kernel,
    apply_synthetic_blur,
    prepare_synthetic_dataset,
    DeblurDataset,
)
from models import build_model, DeblurLoss
from metrics import calculate_psnr, calculate_ssim, calculate_sharpness, batch_psnr_ssim
from evaluate import classical_deblur_baseline

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using Compute Device: {device} (PyTorch {torch.__version__})')

## 1. Visualizing Blur Point-Spread Function (PSF) Kernels & Degradation
Let us visualize the 2D Point Spread Functions (PSF) for **Directional Motion Blur** ($45^\circ$ and $135^\circ$ trajectories) and **Circular Lens Defocus Blur**.

In [ ]:
k_motion_45 = create_motion_blur_kernel(kernel_size=13, angle=45)
k_motion_135 = create_motion_blur_kernel(kernel_size=13, angle=135)
k_defocus = create_defocus_disk_kernel(radius=5)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
for ax, kernel, title in zip(
    axes,
    [k_motion_45, k_motion_135, k_defocus],
    ['Motion PSF (13x13, 45°)', 'Motion PSF (13x13, 135°)', 'Defocus Disk PSF (R=5)'],
):
    ax.imshow(kernel, cmap='inferno')
    ax.set_title(title, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Dataset Preparation & Sample Inspection
We load our paired `(blurred, sharp)` dataset splits (`train`, `val`, `test`). If you have a custom dataset (such as GoPro or DIV2K), place the images inside `data/train/blurred` and `data/train/sharp`.

In [ ]:
data_dir = PROJECT_DIR / 'data'
prepare_synthetic_dataset(data_dir, num_train=96, num_val=24, num_test=18, img_size=96)

train_ds = DeblurDataset(data_dir / 'train', patch_size=96, augment=True)
val_ds = DeblurDataset(data_dir / 'val', patch_size=96, augment=False)
test_ds = DeblurDataset(data_dir / 'test', patch_size=None, augment=False)

print(f'Train Pairs: {len(train_ds)} | Val Pairs: {len(val_ds)} | Test Pairs: {len(test_ds)}')

# Visualize a sample training pair
blur_t, sharp_t, fname = test_ds[0]
blur_img = blur_t.permute(1, 2, 0).numpy()
sharp_img = sharp_t.permute(1, 2, 0).numpy()

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(blur_img)
ax[0].set_title(f'Blurred Input ({fname})\nPSNR: {calculate_psnr(blur_img, sharp_img):.2f} dB | SSIM: {calculate_ssim(blur_img, sharp_img):.3f}')
ax[0].axis('off')
ax[1].imshow(sharp_img)
ax[1].set_title('Ground Truth Sharp Image')
ax[1].axis('off')
plt.tight_layout()
plt.show()

## 3. Model Architecture (`DeblurUNet`) & Hybrid Loss Function
We instantiate the **Residual Channel-Attention U-Net** and inspect its parameter count and structure.

In [ ]:
model = build_model(arch='unet', base_channels=24).to(device)
criterion = DeblurLoss().to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Architecture: DeblurUNet | Trainable Parameters: {total_params:,}')

## 4. Training Curves & Convergence Analysis
Let us inspect the training and validation Loss, PSNR (dB), and SSIM recorded across epochs.

In [ ]:
history_csv = PROJECT_DIR / 'results' / 'training_history.csv'
curves_img = PROJECT_DIR / 'results' / 'training_curves.png'

if history_csv.exists():
    df_hist = pd.read_csv(history_csv)
    display(df_hist.tail(5))
if curves_img.exists():
    img = plt.imread(str(curves_img))
    plt.figure(figsize=(15, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

## 5. Quantitative & Qualitative Evaluation on Unseen Test Images
We load the best trained checkpoint (`checkpoints/best_deblur_model.pth`) and compare:
1. **Blurred Input**
2. **Classical Bilateral + Unsharp Filter**
3. **Deep Learning `DeblurUNet` (Ours)**
4. **Ground Truth Sharp Reference**

In [ ]:
ckpt_path = PROJECT_DIR / 'checkpoints' / 'best_deblur_model.pth'
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model = build_model(arch=ckpt.get('arch', 'unet'), base_channels=ckpt.get('base_channels', 24)).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    print(f"Loaded checkpoint from Epoch {ckpt['epoch']} (Val PSNR: {ckpt['val_psnr']:.2f} dB, Val SSIM: {ckpt['val_ssim']:.4f})")

eval_csv = PROJECT_DIR / 'results' / 'evaluation_metrics.csv'
if eval_csv.exists():
    df_eval = pd.read_csv(eval_csv)
    display(df_eval[['filename', 'blur_type', 'blurred_psnr', 'classical_psnr', 'dl_psnr', 'blurred_ssim', 'dl_ssim', 'psnr_gain']].head(10))

vis_img = PROJECT_DIR / 'results' / 'visual_comparison.png'
if vis_img.exists():
    plt.figure(figsize=(14, 12))
    plt.imshow(plt.imread(str(vis_img)))
    plt.axis('off')
    plt.show()